# p622 Design Circular Queue 解析笔记

- 题号：p622
- 题目英文名：Design Circular Queue
- 题目中文名：设计循环队列
- 题目链接：https://leetcode.cn/problems/design-circular-queue/
- 题型：队列设计
- 难度：Medium
- 推荐优先级：低
- Java 原代码位置：`solutions-java/leetcode/p622_design_circular_queue/DesignCircularQueue.java`


## 1. 题目一句话总结

这道题要求我们用固定长度数组实现一个循环队列。

本质上考的是如何用下标回绕来复用数组空间，以及如何正确维护头尾指针和当前元素个数。


## 2. 题目理解与约束分析

### 2.1 题目要求

实现一个固定容量为 `k` 的循环队列，支持入队、出队、查看队首、查看队尾、判空、判满。

### 2.2 输入与输出

- 输入：一系列队列操作
- 输出：对应操作结果
- 返回结果含义：行为必须符合循环队列定义

### 2.3 关键约束

- 容量固定
- 队列尾部到数组末端后，需要回到下标 `0`
- 不能只靠 `head == tail` 判断空或满，必须结合额外状态

### 2.4 示例分析

如果容量是 `3`，执行：

```text
enQueue(1), enQueue(2), enQueue(3)
deQueue()
enQueue(4)
```

那么 `4` 可以复用之前被弹出的数组位置，这就是“循环”的核心。


## 3. Java 原代码

```java
package leetcode.p622_design_circular_queue;

public class DesignCircularQueue {

    public int[] queue;
    public int head;
    public int tail;
    public int size;
    public int limit;

    public DesignCircularQueue(int n) {
        queue = new int[n];
        head = 0;
        tail = 0;
        size = 0;
        limit = n;
    }

    public boolean enQueue(int value) {
        if (isFull()) {
            return false;
        } else {
            queue[tail] = value;
            tail = (tail + 1 == limit) ? 0 : tail + 1;
            size++;
            return true;
        }
    }

    public boolean deQueue() {
        if (isEmpty()) {
            return false;
        } else {
            queue[head] = 0;
            head = (head + 1 == limit) ? 0 : head + 1;
            size--;
            return true;
        }
    }

    public int Front() {
        return isEmpty() ? -1 : queue[head];
    }

    public int Rear() {
        if (isEmpty()) {
            return -1;
        } else {
            int rearIndex = tail == 0 ? limit - 1 : tail - 1;
            return queue[rearIndex];
        }
    }

    public boolean isEmpty() {
        return size == 0;
    }

    public boolean isFull() {
        return size == limit;
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案直接用固定数组配合四个状态量：

- `head`：队首下标
- `tail`：下一次入队的位置
- `size`：当前元素个数
- `limit`：容量

最核心的地方是头尾指针的回绕：

```java
tail = (tail + 1 == limit) ? 0 : tail + 1;
head = (head + 1 == limit) ? 0 : head + 1;
```

这让数组末端之后可以自然回到开头继续使用。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最简单当然是普通队列，但题目明确要求实现循环队列。

### 5.2 为什么不能简单数组顺移

- 每次出队后整体搬移数组会变成 `O(n)`
- 浪费已有空位

### 5.3 优化方向

Java 原方案通过“指针回绕 + 记录 size”避免了数组搬移，实现了真正的 `O(1)` 入队和出队。


## 6. 核心算法讲解

### 6.1 算法名称

- 固定数组循环队列
- 指针回绕

### 6.2 为什么想到这个算法

因为循环队列最关键的是复用数组空间，而不是物理移动元素。只要维护好头尾下标和元素数量，就可以在逻辑上形成一个环。

### 6.3 关键状态

- `queue`：底层数组
- `head`：队首位置
- `tail`：下一个入队位置
- `size`：当前元素个数
- `limit`：数组容量

### 6.4 步骤拆解

1. 入队时先检查是否已满
2. 把值写到 `tail` 位置
3. `tail` 向前一格，必要时回到 `0`
4. 出队时先检查是否为空
5. `head` 向前一格，必要时回到 `0`
6. `Front` 取 `head` 位置，`Rear` 取 `tail` 前一格位置


In [ ]:
class MyCircularQueue:
    def __init__(self, k: int):
        self.queue = [0] * k
        self.head = 0
        self.tail = 0
        self.size = 0
        self.limit = k

    def enQueue(self, value: int) -> bool:
        if self.isFull():
            return False
        self.queue[self.tail] = value
        self.tail = 0 if self.tail + 1 == self.limit else self.tail + 1
        self.size += 1
        return True

    def deQueue(self) -> bool:
        if self.isEmpty():
            return False
        self.queue[self.head] = 0
        self.head = 0 if self.head + 1 == self.limit else self.head + 1
        self.size -= 1
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.queue[self.head]

    def Rear(self) -> int:
        if self.isEmpty():
            return -1
        rear_index = self.limit - 1 if self.tail == 0 else self.tail - 1
        return self.queue[rear_index]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.limit


## 8. 代码逐段讲解

### 8.1 `enQueue`

入队时写入 `tail`，然后让 `tail` 前进一格。Java 原解里 `tail` 永远指向“下一次可写入位置”，不是当前队尾元素本身。

### 8.2 `deQueue`

出队时并不需要整体搬移，只需要让 `head` 前进即可。

### 8.3 `Rear`

因为 `tail` 指向的是“下一次入队位置”，所以真正的队尾元素在 `tail - 1`，如果 `tail == 0`，就要回绕到 `limit - 1`。


## 9. 复杂度分析

- 时间复杂度：所有操作都是 `O(1)`
- 空间复杂度：`O(k)`
- 为什么是这个空间复杂度：底层固定数组长度就是容量 `k`


## 10. 易错点

1. 只靠 `head == tail` 判断空满，结果空和满状态混淆。
2. `tail` 的语义搞错，以为它总是指向当前队尾。
3. 回绕条件写错，数组越界。
4. `Rear()` 直接返回 `queue[tail]`，这是错的，因为 `tail` 指向的是下一次入队位置。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用固定数组来做，并维护 `head`、`tail`、`size` 三个状态量。
> 其中 `tail` 指向下一个入队位置，`head` 指向当前队首位置。
> 入队和出队都只移动指针，不搬移数据，并且在到达数组末尾后回绕到开头。
> 这样所有操作都能做到 `O(1)`。


In [ ]:
q = MyCircularQueue(3)
print(q.enQueue(1))
print(q.enQueue(2))
print(q.enQueue(3))
print(q.enQueue(4))
print(q.Rear())
print(q.isFull())
print(q.deQueue())
print(q.enQueue(4))
print(q.Rear())


## 12. Demo 输出说明

- 第四次入队应失败，说明满队列判断正确。
- 出队后再次入队 `4` 应成功，说明数组空间被循环复用了。
- 最后队尾应为 `4`。


## 13. 一句话复盘

> 这题最关键的不是数组存数据，而是像 Java 原解那样清楚地区分 `head`、`tail` 和 `size` 的角色。
